<a href="https://colab.research.google.com/github/JulianNguyen05/xiangqi-sorting-jetmax/blob/main/training/train_yolo11.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Chuẩn bị môi trường
Kiểm tra GPU, mount Drive và cài ultralytics.

In [ ]:
import torch
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'KHÔNG CÓ GPU - đổi runtime sang T4')

from google.colab import drive
drive.mount('/content/drive')

!pip install -q ultralytics

## Giải nén dataset hợp nhất
Copy merged_v1.zip từ Drive về máy Colab (đọc nhanh hơn đọc trực tiếp từ Drive) và kiểm tra số ảnh/nhãn mỗi tập.

In [ ]:
import os, shutil

BASE = '/content/drive/MyDrive/datn'
MERGED = '/content/merged'

if not os.path.isdir(f'{MERGED}/train'):
    shutil.unpack_archive(f'{BASE}/merged/merged_v1.zip', MERGED)

for s in ['train', 'val', 'test']:
    print(s, len(os.listdir(f'{MERGED}/{s}/images')), 'ảnh |', len(os.listdir(f'{MERGED}/{s}/labels')), 'nhãn')

print(open(f'{MERGED}/data.yaml').read())

## Kiểm tra nhãn trước khi huấn luyện
Đảm bảo mỗi ảnh đều có file nhãn tương ứng và mọi class id nằm trong khoảng 0-13. Nhãn sai class id sẽ làm quá trình train báo lỗi hoặc cho kết quả sai.

In [ ]:
import glob

bad = 0
for s in ['train', 'val', 'test']:
    imgs = {os.path.splitext(f)[0] for f in os.listdir(f'{MERGED}/{s}/images')}
    lbls = {os.path.splitext(f)[0] for f in os.listdir(f'{MERGED}/{s}/labels')}
    print(s, '| ảnh thiếu nhãn:', len(imgs - lbls), '| nhãn thiếu ảnh:', len(lbls - imgs))
    for p in glob.glob(f'{MERGED}/{s}/labels/*.txt'):
        for line in open(p):
            parts = line.split()
            if parts and not (0 <= int(parts[0]) <= 13):
                bad += 1
print('Dòng nhãn có class id sai:', bad)

## Khai báo tham số huấn luyện
Gom các siêu tham số vào một chỗ để dễ chỉnh. Kết quả được lưu thẳng vào Drive (`/datn/runs`) nên không mất khi Colab ngắt kết nối. Tên run `yolo11n_640` khớp với thư mục `runs/` trong dự án.

In [ ]:
from ultralytics import YOLO

DATA    = f'{MERGED}/data.yaml'
PROJECT = f'{BASE}/runs'
NAME    = 'yolo11n_640'

EPOCHS  = 100
IMGSZ   = 640
BATCH   = 32

## Nạp mô hình YOLO11n pretrained
Dùng trọng số `yolo11n.pt` đã huấn luyện sẵn trên COCO (transfer learning). Bản `n` (nano) nhỏ và nhanh, phù hợp triển khai lên Jetson Nano. Trọng số sẽ tự tải về lần đầu.

In [ ]:
model = YOLO('yolo11n.pt')

## Huấn luyện mô hình
Các tham số đáng chú ý:
- `patience=30`: dừng sớm nếu 30 epoch liền không cải thiện.
- `fliplr=0.0`: **tắt lật ngang** vì lật sẽ làm chữ Hán trên quân cờ bị đảo ngược, gây nhầm lớp.
- `degrees=10`: xoay nhẹ, vì quân cờ trên băng chuyền có thể nằm lệch góc.
- `save_period=10`: lưu checkpoint mỗi 10 epoch để dự phòng khi Colab ngắt.
- `close_mosaic=10`: tắt mosaic ở 10 epoch cuối để mô hình ổn định hơn.

Thời gian ước tính khoảng 1-2 giờ trên T4.

In [ ]:
results = model.train(
    data=DATA,
    epochs=EPOCHS,
    imgsz=IMGSZ,
    batch=BATCH,
    patience=30,
    project=PROJECT,
    name=NAME,
    exist_ok=True,
    save_period=10,
    workers=2,
    fliplr=0.0,
    degrees=10,
    close_mosaic=10,
    plots=True,
)

Ultralytics 8.4.177 🚀 Python-3.13.15 torch-2.11.0+cu130 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/merged/data.yaml, degrees=10, deterministic=True, device=, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=100, erasing=0.4, exist_ok=True, fliplr=0.0, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, iou=0.7, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolo11n.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=yolo11n_640, nbs=64, nms=None, opset=None, optimize=False, optimi

## Tiếp tục huấn luyện nếu bị ngắt giữa chừng
Chỉ chạy ô này khi Colab ngắt kết nối lúc đang train. Chạy lại phần chuẩn bị môi trường và giải nén dataset, bỏ qua bước 4, rồi chạy ô dưới để train tiếp từ checkpoint `last.pt`.

In [ ]:
# model = YOLO(f'{PROJECT}/{NAME}/weights/last.pt')
# model.train(resume=True)

## Xem kết quả huấn luyện
Đọc `results.csv` và vẽ đường loss cùng mAP theo từng epoch. Loss giảm đều và mAP tăng rồi đi ngang là dấu hiệu huấn luyện tốt. Nếu loss val tăng trong khi loss train giảm thì mô hình đang overfit.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

df = pd.read_csv(f'{PROJECT}/{NAME}/results.csv')
df.columns = df.columns.str.strip()

fig, ax = plt.subplots(1, 2, figsize=(13, 4))
for c in ['train/box_loss', 'val/box_loss', 'train/cls_loss', 'val/cls_loss']:
    ax[0].plot(df['epoch'], df[c], label=c)
ax[0].set_title('Loss'); ax[0].set_xlabel('epoch'); ax[0].legend()

for c in ['metrics/mAP50(B)', 'metrics/mAP50-95(B)']:
    ax[1].plot(df['epoch'], df[c], label=c)
ax[1].set_title('mAP (val)'); ax[1].set_xlabel('epoch'); ax[1].legend()
plt.show()

print(df[['epoch', 'metrics/precision(B)', 'metrics/recall(B)',
          'metrics/mAP50(B)', 'metrics/mAP50-95(B)']].tail(3))

## Đánh giá trên tập test
Tập test chưa từng được dùng khi huấn luyện hay chọn mô hình, nên đây là con số khách quan để báo cáo. Mục tiêu trong đề cương là **mAP@0.5 ≥ 95%**. Bảng bên dưới cho biết từng lớp để phát hiện lớp nào còn yếu.

In [ ]:
best = YOLO(f'{PROJECT}/{NAME}/weights/best.pt')
metrics = best.val(data=DATA, split='test', imgsz=IMGSZ, project=PROJECT, name=f'{NAME}_test', exist_ok=True)

print(f'mAP@0.5     : {metrics.box.map50:.4f}')
print(f'mAP@0.5:0.95: {metrics.box.map:.4f}')
print(f'Precision   : {metrics.box.mp:.4f} | Recall: {metrics.box.mr:.4f}')

print('\nmAP@0.5 theo lớp:')
for i, name in best.names.items():
    print(f'  {name:15s} {metrics.box.ap50[i]:.4f}')

## Dự đoán thử trên ảnh test
Chọn ngẫu nhiên vài ảnh từ tập test để xem mô hình khoanh vùng và gán nhãn có hợp lý không. Bước này hữu ích để đưa vào báo cáo.

In [ ]:
import random
from IPython.display import Image, display

samples = random.sample(glob.glob(f'{MERGED}/test/images/*'), 4)
preds = best.predict(samples, imgsz=IMGSZ, conf=0.5, save=True,
                     project=PROJECT, name=f'{NAME}_pred', exist_ok=True)

for r in preds:
    display(Image(filename=os.path.join(r.save_dir, os.path.basename(r.path))))

## Xuất mô hình sang ONNX
`best.pt` dùng được ngay với Ultralytics, còn ONNX là định dạng trung gian để chuyển sang TensorRT. **Lưu ý:** file TensorRT (`.engine`) phải được tạo trực tiếp trên Jetson, không tạo trên Colab được, vì nó phụ thuộc vào GPU và phiên bản TensorRT của thiết bị.

In [ ]:
onnx_path = best.export(format='onnx', imgsz=IMGSZ, simplify=True)
print('Đã xuất:', onnx_path)
print('Các file trong thư mục weights:', os.listdir(f'{PROJECT}/{NAME}/weights'))

## Tải kết quả về máy
Nén thư mục kết quả và tải về, giải nén vào `runs/yolo11n_640/` trong dự án. Bản gốc vẫn còn trên Drive.

In [ ]:
from google.colab import files

shutil.make_archive('/content/yolo11n_640', 'zip', f'{PROJECT}/{NAME}')
files.download('/content/yolo11n_640.zip')